# Load dim_product 

Starting the notebook with %run "/Workspace/Shared/notebook_init"  loads common variables and constants used across all notebooks for the Vinoworld project

JDD TEST EDIT

CATALOG, BRONZE, SILVER, GOLD, AUDIT, RAW_FILES,
PIPELINE_RUN_ID, Utils, F, Row, datetime etc.

import uuid, time
from datetime import datetime, timezone
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp, lit, input_file_name, col
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, IntegerType
sys.path.append("/Workspace/Shared")

import pipeline_utils as Utils
from pipeline_logging import pipeline_log_upsert, pipeline_step_log_upsert, ingestion_log_insert


In [0]:
%run "../../libs/notebook_init"

In [0]:
# Imports and constants specific to the dim_product Silver SCD2 load.
# SOURCE_SUBPATH is the subfolder under RAW_FILES (legacy — not used here);
# TARGET_TABLE is the fully-qualified Silver table name.

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from datetime import datetime, timezone
import traceback

# transform_detail_log_insert isn't in notebook_init's central import yet —
# pull it in here so this notebook can log per-transform audit rows.
from pipeline_logging import transform_detail_log_insert

%load_ext autoreload
%autoreload 2

SOURCE_SUBPATH = "productdata"                       # case must match volume
SOURCE_PATH    = f"{RAW_FILES}{SOURCE_SUBPATH}"
TARGET_TABLE   = f"{SILVER}.dim_product"

# print(f"Products Source Path:  {SOURCE_PATH}")


In [0]:
# ----------------------------------------------------------------------
# Setup the variables need for initial call to pipeline_step_log_upsert
# ----------------------------------------------------------------------

nb = Utils.get_notebook_context(dbutils)
notebook_folder = nb['notebook_folder']
notebook_name = nb['notebook_name']

#logger.info(f"Inserting pipeline_step_log record for notebook {notebook_name}")

step_log_id       = str(uuid.uuid4())
pipeline_run_id   = PIPELINE_RUN_ID
step_sequence     = 1
layer             = "silver"
target_table      = TARGET_TABLE
status            = "running"
started_timestamp = datetime.now(timezone.utc)
rows_read         = 0
rows_written      = 0
error_message     = None


pipeline_step_log_upsert(spark, step_log_id, pipeline_run_id, step_sequence, notebook_folder, notebook_name, status, started_timestamp, layer, target_table)

# All Parameters
# pipeline_step_log_upsert(spark, step_log_id, pipeline_run_id, step_sequence, notebook_folder, notebook_name, status, started_timestamp, layer, target_table, rows_read, rows_written,  ended_timestamp,  error_message)

In [0]:
## ============================================================
#  Pipeline: vinoworld.bronze.products → vinoworld.silver.dim_product
#  Pattern: Type 2 SCD via multi-statement approach
#  Databricks Delta Lake (Free / Standard tier)
#  ============================================================

#  ────────────────────────────────────────────────────────────
#  STEP 1: Stage incoming data — cast, filter, deduplicate,
#          and compute the RowHash for change detection.
# ────────────────────────────────────────────────────────────

try:
    # Defining the SQL logic using the optimized QUALIFY clause for idempotency
    sql_query = f"""
        CREATE OR REPLACE TEMPORARY VIEW vw_staged_products AS
        SELECT
            product_no                                  AS ProductNo,
            title                                       AS ProductName,
            COALESCE(province, 'Unknown')               AS Province,
            region_1                                    AS Region,
            variety                                     AS Variety,
            winery                                      AS Winery,
            TRY_CAST(vintage AS SMALLINT)               AS Vintage,
            COALESCE(TRY_CAST(score AS SMALLINT), 0)    AS Score,
            COALESCE(TRY_CAST(CAST(CAST(dealer_price AS DECIMAL(10,2)) AS INT) AS INT), 0)
                                                        AS DealerPrice,
            COALESCE(TRY_CAST(markup AS DECIMAL(5,2)), 0.00)
                                                        AS Markup,
            COALESCE(TRY_CAST(CAST(CAST(list_price AS DECIMAL(10,2)) AS INT) AS INT), 0)
                                                        AS ListPrice,
            MD5(CONCAT_WS('|',
                COALESCE(title,        ''),
                COALESCE(province,     ''),
                COALESCE(region_1,     ''),
                COALESCE(variety,      ''),
                COALESCE(winery,       ''),
                COALESCE(vintage,      ''),
                COALESCE(score,        ''),
                COALESCE(dealer_price, ''),
                COALESCE(markup,       ''),
                COALESCE(list_price,   '')
            ))                                          AS RowHash
        FROM {CATALOG}.bronze.products
        WHERE title IS NOT NULL
        AND product_no IS NOT NULL
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY product_no
            ORDER BY inserted_ts DESC
        ) = 1
    """

    # Execute the query
    spark.sql(sql_query)
    print("Success: Temporary view 'vw_staged_products' created.")


except Exception as e:
    err = Utils.capture_exception(e)
    error_message = f"{err['error_type']}: {err['error_message']}\n\n{err['error_traceback']}"
    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_FAILED
    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, 0, ended_timestamp, error_message
    )
    raise

In [0]:
# Per-transform variables for transform_detail_log. Defined OUTSIDE the try
# block so the except handler can log a failed transform row even if the
# atomic SQL never started.
transform_source_table = f"{CATALOG}.bronze.products"
transform_target_table = f"{CATALOG}.silver.dim_product"
transform_started      = datetime.now(timezone.utc)
rows_inserted          = 0
rows_expired           = 0

try:
    # Pre-counts so we can derive rows_inserted / rows_expired.
    # DESCRIBE HISTORY returns zeros for BEGIN ATOMIC blocks, so we use a
    # pre/post count differential instead.
    pre_current_count = spark.sql(
        f"SELECT COUNT(*) FROM {transform_target_table} WHERE IsRowCurrent = TRUE"
    ).collect()[0][0]
    pre_total_count   = spark.table(transform_target_table).count()
    rows_read = spark.sql("SELECT COUNT(*) FROM vw_staged_products").collect()[0][0]

    # We wrap both statements in a single BEGIN ATOMIC block
    # and pass it to a single spark.sql() call.
    atomic_scd2_sql = f"""
    BEGIN ATOMIC
        -- STEP 2: Expire current silver rows whose hash has changed
        MERGE INTO {CATALOG}.silver.dim_product AS tgt
        USING (
            SELECT s.ProductNo, s.RowHash
            FROM   vw_staged_products      s
            JOIN   {CATALOG}.silver.dim_product d
                ON  d.ProductNo    = s.ProductNo
                AND d.IsRowCurrent = TRUE
                AND d.RowHash     <> s.RowHash
        ) AS src
        ON  tgt.ProductNo    = src.ProductNo
        AND tgt.IsRowCurrent = TRUE
        WHEN MATCHED THEN UPDATE SET
            tgt.IsRowCurrent = FALSE,
            tgt.EndDate      = CURRENT_TIMESTAMP(),
            tgt.UpdatedDate  = CURRENT_TIMESTAMP();

        -- STEP 3: Insert new rows
        INSERT INTO {CATALOG}.silver.dim_product (
            ProductNo, ProductName, Province, Region, Variety, Winery,
            Vintage, Score, DealerPrice, Markup, ListPrice, RowHash,
            IsRowCurrent, EffectiveDate, EndDate, UpdatedDate
        )
        SELECT
            s.ProductNo, s.ProductName, s.Province, s.Region, s.Variety, s.Winery,
            s.Vintage, s.Score, s.DealerPrice, s.Markup, s.ListPrice, s.RowHash,
            TRUE                                        AS IsRowCurrent,
            CURRENT_TIMESTAMP()                         AS EffectiveDate,
            CAST('9999-12-31 00:00:00' AS TIMESTAMP)    AS EndDate,
            CURRENT_TIMESTAMP()                         AS UpdatedDate
        FROM vw_staged_products s
        WHERE NOT EXISTS (
            SELECT 1
            FROM {CATALOG}.silver.dim_product d
            WHERE d.ProductNo    = s.ProductNo
              AND d.IsRowCurrent = TRUE
        );
    END;
    """

    print("Executing atomic SCD Type 2 transaction...")
    spark.sql(atomic_scd2_sql)
    print("Success: Transaction committed.")

    # Post-counts → SCD2 row decomposition.
    post_current_count = spark.sql(
        f"SELECT COUNT(*) FROM {transform_target_table} WHERE IsRowCurrent = TRUE"
    ).collect()[0][0]
    post_total_count   = spark.table(transform_target_table).count()
    rows_inserted = post_total_count - pre_total_count
    rows_expired  = pre_current_count - (post_current_count - rows_inserted)
    rows_written  = rows_inserted + rows_expired

    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_SUCCEEDED

    transform_detail_log_insert(
        spark,
        pipeline_run_id   = PIPELINE_RUN_ID,
        step_log_id       = step_log_id,
        source_table      = transform_source_table,
        target_table      = transform_target_table,
        status            = status,
        started_timestamp = transform_started,
        ended_timestamp   = ended_timestamp,
        rows_read         = rows_read,
        rows_written      = rows_written,
        rows_inserted     = rows_inserted,
        rows_expired      = rows_expired,
    )

    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, rows_written, ended_timestamp, error_message
    )

except Exception as e:
    err = Utils.capture_exception(e)
    error_message = f"{err['error_type']}: {err['error_message']}\n\n{err['error_traceback']}"
    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_FAILED

    transform_detail_log_insert(
        spark,
        pipeline_run_id   = PIPELINE_RUN_ID,
        step_log_id       = step_log_id,
        source_table      = transform_source_table,
        target_table      = transform_target_table,
        status            = status,
        started_timestamp = transform_started,
        ended_timestamp   = ended_timestamp,
        rows_read         = rows_read,
        rows_inserted     = rows_inserted,
        rows_expired      = rows_expired,
        error_message     = error_message,
    )

    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, 0, ended_timestamp, error_message
    )
    raise


In [0]:

try:


    count = spark.sql(f"""
        SELECT COUNT(*) as cnt
        FROM {CATALOG}.silver.dim_product
        WHERE ProductNo = '-1'
    """).collect()[0]['cnt']


    if count == 0:
        spark.sql(f"""
            INSERT INTO {CATALOG}.silver.dim_product (
            ProductNo, ProductName, Province, Region, Variety, Winery,
            Vintage, Score, DealerPrice, Markup, ListPrice, RowHash,
            IsRowCurrent, EffectiveDate, EndDate, UpdatedDate
        )
        VALUES (
            '-1', 'Unknown', 'Unknown', 'Unknown', 'Unknown', 'Unknown',
            CAST(-1 AS SMALLINT),
            CAST(-1 AS SMALLINT),
            -1, -1.00, -1,
            md5(concat_ws('|',
                'Unknown',   -- title
                'Unknown',   -- province
                'Unknown',   -- region_1
                'Unknown',   -- variety
                'Unknown',   -- winery
                '0',        -- Vintage
                '0',        -- Score
                '0',        -- DealerPrice
                '0.00',     -- Markup
                '0'         -- ListPrice
            )),
            TRUE,
            current_timestamp(),
            CAST('9999-12-31 00:00:00' AS TIMESTAMP),
            current_timestamp()
        )
        """)


except Exception as e:
    err = Utils.capture_exception(e)
    error_message = f"{err['error_type']}: {err['error_message']}\n\n{err['error_traceback']}"
    ended_timestamp = datetime.now(timezone.utc)
    status = STATUS_FAILED
    pipeline_step_log_upsert(
        spark, step_log_id, pipeline_run_id, step_sequence,
        notebook_folder, notebook_name, status, started_timestamp,
        layer, target_table, rows_read, 0, ended_timestamp, error_message
    )
    raise

In [0]:
%skip


metrics = spark.sql(f"DESCRIBE HISTORY {CATALOG}.silver.dim_product LIMIT 1") \
               .select("operationMetrics") \
               .collect()[0][0]

rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
rows_updated  = int(metrics.get("numTargetRowsUpdated",  0))
rows_deleted  = int(metrics.get("numTargetRowsDeleted",  0))

print(f" Rows  Inserted: {rows_inserted:,}")
print(f" Rows     Updated: {rows_updated:,}"   )


In [0]:
%skip
%sql
select * from vinoworld.silver.dim_product
where ProductNo in  ("120", "130", "99999", "361", "1561", "1563")
order by ProductNo